# AIC 2026 B02 ASR worker

Same three-cell workflow as B01: **1. Settings → 2. Mount/setup → 3. Run/resume**. Pinned large-v3-turbo, Vietnamese, checkpoint per video. CPU is temporary while GPU quota is unavailable. Do not run a duplicate worker.

B02 currently has videos only: transcripts retain timestamps, with no invented source-frame IDs. This notebook never finalizes or imports artifacts and never modifies B01. Do not change device or worker count after extraction starts without auditing existing checkpoints.

In [ ]:
# Cell 1 — settings
WORKER_ID = 0
WORKER_COUNT = 1
ASR_PROFILE = 'production'
BATCH_ID = 'aic26-b2'
DEVICE = 'cpu'  # use 'cuda' only after GPU allocation and checkpoint audit
MAX_NEW_VIDEOS = None  # 1 for a sample; None for the assigned lane
assert DEVICE in {'cpu', 'cuda'} and WORKER_ID in range(WORKER_COUNT)


In [ ]:
# Cell 2 — mount Drive and prepare the frozen B02 worker
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AIC2026/B02')
RAW_ROOT = DRIVE_ROOT / 'raw'
OUTPUT_ROOT = DRIVE_ROOT / 'out' / ('asr-video-only-cpu-rerun-v1' if DEVICE == 'cpu' else 'asr-video-only-gpu-v1')
SOURCE_SHA256 = 'e3beb8415fdec14644df11e7110db7a3b725362f988331e3253b77afd1118277'
SOURCE_BUNDLE = DRIVE_ROOT / 'source' / 'b02-asr-e3beb8415fde.zip'
REPO_ROOT = Path('/content/aic2026-b02-asr')
SCRATCH_ROOT = Path('/content/b02-asr-scratch')
PLAN_PATH = OUTPUT_ROOT / 'plan.json'
assert RAW_ROOT.is_dir(), 'Mount the Drive containing AIC2026/B02/raw'
assert SOURCE_BUNDLE.is_file(), 'Missing frozen B02 source ZIP in B02/source'
assert hashlib.sha256(SOURCE_BUNDLE.read_bytes()).hexdigest() == SOURCE_SHA256, 'Source hash mismatch'
assert not subprocess.run(['pgrep', '-af', '[c]olab_video_asr.py'], capture_output=True, text=True).stdout.strip(), 'An ASR worker is already running; do not run setup again'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_BUNDLE) as source:
    for member in source.infolist():
        target = (REPO_ROOT / member.filename).resolve()
        assert REPO_ROOT.resolve() in target.parents, 'Unsafe source member'
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with source.open(member) as inp, target.open('wb') as out:
                shutil.copyfileobj(inp, out)
os.chdir(REPO_ROOT)
def run_visible(args):
    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, args)
    except BaseException:
        if process.poll() is None:
            import signal
            process.send_signal(signal.SIGINT)
            process.wait()
        raise
run_visible([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.16'])
run_visible(['uv', 'python', 'install', '3.11'])
run_visible(['uv', 'sync', '--python', '3.11', '--group', 'competition', '--frozen'])
if DEVICE == 'cuda':
    run_visible(['uv', 'run', '--frozen', 'python', '-c', 'import torch; assert torch.cuda.is_available(), "GPU not allocated"; print(torch.cuda.get_device_name(0))'])
command = ['uv', 'run', '--frozen', 'python', 'scripts/colab_video_asr.py',
    '--raw-dir', str(RAW_ROOT), '--output-dir', str(OUTPUT_ROOT),
    '--batch-id', BATCH_ID, '--worker-count', str(WORKER_COUNT), '--worker-id', str(WORKER_ID),
    '--ignore-empty', 'Videos_M010.zip', '--source-sha256', SOURCE_SHA256,
    '--scratch-dir', str(SCRATCH_ROOT), '--device', DEVICE, '--resume']
run_visible(command + ['--inventory-only'])
run_visible(['uv', 'run', '--frozen', 'python', 'scripts/prepare_models.py', '--only', 'asr'])
print('READY', DEVICE, ASR_PROFILE, str(OUTPUT_ROOT), flush=True)


In [ ]:
# Cell 3 — validate lane, run/resume, show durable progress
import threading

plan = json.loads(PLAN_PATH.read_text())
assert plan['contract'] == 'video-only-asr-staging-v1'
assert plan['batch_id'] == BATCH_ID and plan['worker_count'] == WORKER_COUNT
assigned = [v for v in plan['videos'] if v['worker_id'] == WORKER_ID]
assert assigned, 'Empty worker lane'
print(json.dumps({'plan_fingerprint': plan['fingerprint'], 'worker_id': WORKER_ID,
    'expected_videos': plan['expected_videos'], 'assigned_videos': len(assigned), 'device': DEVICE}), flush=True)
stop_progress = threading.Event()
def show_progress():
    while not stop_progress.wait(60):
        p = OUTPUT_ROOT / 'progress' / f'worker-{WORKER_ID}.json'
        if p.is_file():
            try:
                print('PROGRESS', p.read_text(), flush=True)
            except OSError as error:
                print('Progress read unavailable:', type(error).__name__, flush=True)
progress_thread = threading.Thread(target=show_progress, daemon=True)
progress_thread.start()
try:
    run_visible(command + (['--max-videos', str(MAX_NEW_VIDEOS)] if MAX_NEW_VIDEOS is not None else []))
finally:
    stop_progress.set()
    progress_thread.join(timeout=2)
print('Saved receipts:', str(OUTPUT_ROOT / 'receipts'), flush=True)
